In [3]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


In [4]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense,
    Dropout,
    Concatenate
)
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)
print("Libraries imported successfully!")

TensorFlow version: 2.21.0
Libraries imported successfully!


## Load our dataset

In [6]:
df = pd.read_csv(
    "learntwin_student_interactions.csv"
)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset shape: (10000, 10)

Columns:
['student_id', 'attempt', 'question_id', 'concept', 'difficulty', 'correct', 'time_gap_days', 'days_since_practice', 'timestamp', 'knowledge_state']


,student_id,attempt,question_id,concept,difficulty,correct,time_gap_days,days_since_practice,timestamp,knowledge_state
0,S001,1,Q043,RIGHT JOIN,0.724,0,3.007,3.007,3.007,0.3574
1,S001,2,Q452,RIGHT JOIN,0.577,0,1.471,1.471,4.478,0.3103
2,S001,3,Q022,UNION,0.499,1,1.675,6.154,6.154,0.3849
3,S001,4,Q007,WHERE,0.380,1,0.563,6.717,6.717,0.4015
4,S001,5,Q114,STORED PROCEDURES,0.598,1,2.683,9.400,9.400,0.1345


## Sort interactions chronologically

In [8]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"]
)

df = df.sort_values(
    by=["student_id", "timestamp"]
).reset_index(drop=True)

print(df.head(10))

  student_id  attempt question_id            concept  difficulty  correct  \
0       S001        1        Q043         RIGHT JOIN       0.724        0   
1       S001        2        Q452         RIGHT JOIN       0.577        0   
2       S001        3        Q022              UNION       0.499        1   
3       S001        4        Q007              WHERE       0.380        1   
4       S001        5        Q114  STORED PROCEDURES       0.598        1   
5       S001        6        Q210           GROUP BY       0.475        0   
6       S001        7        Q481           TRIGGERS       0.686        0   
7       S001        8        Q331  STORED PROCEDURES       0.731        0   
8       S001        9        Q390           GROUP BY       0.446        1   
9       S001       10        Q334         SUBQUERIES       0.603        0   

   time_gap_days  days_since_practice                     timestamp  \
0          3.007                3.007 1970-01-01 00:00:00.000000003   
1         

In [9]:
print(df.columns.tolist())

['student_id', 'attempt', 'question_id', 'concept', 'difficulty', 'correct', 'time_gap_days', 'days_since_practice', 'timestamp', 'knowledge_state']


## Encode concepts

LSTM cannot directly process strings such as:

JOIN

Keys

Normalization

Subqueries


In [10]:
concept_encoder = LabelEncoder()

df["concept_encoded"] = concept_encoder.fit_transform(
    df["concept"]
)

print("Number of concepts:",
      len(concept_encoder.classes_))

print("\nConcept mapping:")

for i, concept in enumerate(
    concept_encoder.classes_
):
    print(i, "→", concept)

Number of concepts: 20

Concept mapping:
0 → AGGREGATE FUNCTIONS
1 → FOREIGN KEY
2 → GROUP BY
3 → HAVING
4 → INDEXING
5 → INNER JOIN
6 → JOINS
7 → LEFT JOIN
8 → NORMALIZATION
9 → ORDER BY
10 → PRIMARY KEY
11 → RIGHT JOIN
12 → SELECT
13 → STORED PROCEDURES
14 → SUBQUERIES
15 → TRANSACTIONS
16 → TRIGGERS
17 → UNION
18 → VIEWS
19 → WHERE


## Choose sequence features

In [11]:
sequence_features = [
    "concept_encoded",
    "correct",
    "difficulty",
    "time_gap_days",
    "days_since_practice"
]

print(sequence_features)

['concept_encoded', 'correct', 'difficulty', 'time_gap_days', 'days_since_practice']


In [12]:
target = "correct"

## Create training/test sequence logic

Attempts 1–10

      ↓
      
Predict Attempt 11

Attempts 2–11

      ↓
      
Predict Attempt 12


Attempts 3–12

      ↓
      
Predict Attempt 13

In [13]:
SEQUENCE_LENGTH = 10

print(
    "Sequence length:",
    SEQUENCE_LENGTH
)

Sequence length: 10


In [14]:
X_concept = []
X_numeric = []
y = []
target_attempts = []
target_students = []

for student_id, student_data in df.groupby(
    "student_id"
):
    
    student_data = student_data.sort_values(
        "attempt"
    ).reset_index(drop=True)
    
    for i in range(
        SEQUENCE_LENGTH,
        len(student_data)
    ):
        
        # Previous 10 interactions
        history = student_data.iloc[
            i - SEQUENCE_LENGTH:i
        ]
        
        # Current interaction = target
        current = student_data.iloc[i]
        
        # Concept sequence
        X_concept.append(
            history["concept_encoded"].values
        )
        
        # Numerical sequence
        X_numeric.append(
            history[
                [
                    "correct",
                    "difficulty",
                    "time_gap_days",
                    "days_since_practice"
                ]
            ].values
        )
        
        # Current answer
        y.append(
            current["correct"]
        )
        
        # Store target information
        target_attempts.append(
            current["attempt"]
        )
        
        target_students.append(
            current["student_id"]
        )

In [15]:
X_concept = np.array(
    X_concept,
    dtype=np.int32
)

X_numeric = np.array(
    X_numeric,
    dtype=np.float32
)

y = np.array(
    y,
    dtype=np.float32
)

target_attempts = np.array(
    target_attempts
)

target_students = np.array(
    target_students
)

In [16]:
print("Concept sequence shape:",
      X_concept.shape)

print("Numeric sequence shape:",
      X_numeric.shape)

print("Target shape:",
      y.shape)

Concept sequence shape: (9000, 10)
Numeric sequence shape: (9000, 10, 4)
Target shape: (9000,)


## Split sequences without leakage

In [17]:
train_mask = target_attempts <= 80
test_mask = target_attempts > 80

X_concept_train = X_concept[train_mask]
X_concept_test = X_concept[test_mask]

X_numeric_train = X_numeric[train_mask]
X_numeric_test = X_numeric[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

In [18]:
print(
    "Training sequences:",
    len(y_train)
)

print(
    "Testing sequences:",
    len(y_test)
)

Training sequences: 7000
Testing sequences: 2000


## Scale numerical features

In [19]:
scaler = StandardScaler()

num_train_2d = X_numeric_train.reshape(
    -1,
    X_numeric_train.shape[-1]
)

num_test_2d = X_numeric_test.reshape(
    -1,
    X_numeric_test.shape[-1]
)

In [20]:
scaler.fit(num_train_2d)

StandardScaler()

In [21]:
X_numeric_train = scaler.transform(
    num_train_2d
).reshape(
    X_numeric_train.shape
)

X_numeric_test = scaler.transform(
    num_test_2d
).reshape(
    X_numeric_test.shape
)

In [22]:
print(
    "Scaled training shape:",
    X_numeric_train.shape
)

print(
    "Scaled testing shape:",
    X_numeric_test.shape
)

Scaled training shape: (7000, 10, 4)
Scaled testing shape: (2000, 10, 4)


## Check target distribution

In [23]:
print("Training target:")
print(
    pd.Series(y_train)
    .value_counts()
)

print("\nTesting target:")
print(
    pd.Series(y_test)
    .value_counts()
)

Training target:
0.0    4941
1.0    2059
Name: count, dtype: int64

Testing target:
0.0    1413
1.0     587
Name: count, dtype: int64


In [24]:
print("\nTraining percentages:")
print(
    pd.Series(y_train)
    .value_counts(
        normalize=True
    ) * 100
)


Training percentages:
0.0    70.585714
1.0    29.414286
Name: proportion, dtype: float64


                    Previous 10 attempts
                           │
        ┌──────────────────┼──────────────────┐
        ↓                  ↓                  ↓
     Concepts          Correctness       Time information
        │                  │                  │
        ↓                  ↓                  ↓
   Embedding          Numerical data      Numerical data
        └──────────────────┬──────────────────┘
                           ↓
                         LSTM
                           ↓
                       Dense
                           ↓
                  Probability correct